In [2]:
pip install transformers datasets torch accelerate scikit-learn



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import pandas as pd
import numpy as np
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
df = pd.read_csv("/Users/bimsaraserasinghe/Library/Mobile Documents/com~apple~CloudDocs/DSGP/Pre-Processed Csvs/cleaned_tweet_data.csv")
df.head()


,tweet_id,cleaned_content,sentiment
0,1956967341,i know i was listenin to bad habit earlier and...,empty
1,1956967666,layin n bed with a headache ughhhhwaitin on yo...,sadness
2,1956967696,funeral ceremonygloomy friday,sadness
3,1956967789,wants to hang out with friends soon,enthusiasm
4,1956968416,we want to trade with someone who has houston ...,neutral


In [8]:
label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["sentiment"])

num_labels = len(label_encoder.classes_)
print(label_encoder.classes_)


['anger' 'boredom' 'empty' 'enthusiasm' 'fun' 'happiness' 'hate' 'love'
 'neutral' 'relief' 'sadness' 'surprise' 'worry']


In [9]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)


In [10]:
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)


In [11]:
model_name = "j-hartmann/emotion-english-distilroberta-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)


In [14]:
def tokenize_function(examples):
    return tokenizer(
        examples["cleaned_content"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

train_dataset = train_dataset.map(tokenize_function, batched=True)
val_dataset = val_dataset.map(tokenize_function, batched=True)


Map: 100%|██████████| 7947/7947 [00:00<00:00, 52585.54 examples/s]


In [15]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    ignore_mismatched_sizes=True
)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2874.89it/s, Materializing param=roberta.encoder.layer.5.output.dense.weight]             
RobertaForSequenceClassification LOAD REPORT from: j-hartmann/emotion-english-distilroberta-base
Key                             | Status     |                                                                                        
--------------------------------+------------+----------------------------------------------------------------------------------------
roberta.embeddings.position_ids | UNEXPECTED |                                                                                        
classifier.out_proj.weight      | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([7, 768]) vs model:torch.Size([13, 768])
classifier.out_proj.bias        | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([7]) vs model:torch.Size([13])          

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architectur

In [16]:
training_args = TrainingArguments(
    output_dir="./emotion_model",
    eval_strategy="epoch", # Changed from evaluation_strategy
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_dir="./logs",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    report_to="none"
)

`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [17]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1": f1_score(labels, predictions, average="weighted")
    }


In [18]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics
)

In [19]:
trainer.train()


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,1.829436,1.778328,0.403549,0.370644
2,1.708486,1.773608,0.404052,0.374982
3,1.608148,1.787747,0.397760,0.371746


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.52it/s]
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.45it/s]
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  2.03it/s]
There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.

TrainOutput(global_step=5961, training_loss=1.7250880110365816, metrics={'train_runtime': 4722.7451, 'train_samples_per_second': 20.191, 'train_steps_per_second': 1.262, 'total_flos': 3158576044651008.0, 'train_loss': 1.7250880110365816, 'epoch': 3.0})

In [20]:
trainer.save_model("fine_tuned_emotion_model")

import joblib
joblib.dump(label_encoder, "label_encoder.pkl")


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.59it/s]


['label_encoder.pkl']

In [21]:
from transformers import pipeline

emotion_classifier = pipeline(
    "text-classification",
    model="fine_tuned_emotion_model",
    tokenizer=tokenizer
)


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1756.89it/s, Materializing param=roberta.encoder.layer.5.output.dense.weight]             


In [9]:
def emotion_chatbot():
    print("Fine-Tuned Emotion Chatbot")
    print("Type 'exit' to stop\n")

    while True:
        user_input = input("You: ")

        if user_input.lower() == "exit":
            print("Bot: Take care ")
            break

        result = emotion_classifier(user_input)[0]
        label = label_encoder.inverse_transform([int(result["label"].split("_")[-1])])[0]
        print(f"Bot: I sense **{label}**")
